# 04 — Datetime Feature Engineering

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 55 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Extract rich, interpretable features from raw datetime columns
2. Implement **cyclical encoding** (sin/cos) for hour, day-of-week, and month
3. Build **lag features** and **rolling window aggregations** to capture temporal autocorrelation
4. Handle timezones and missing datetimes safely
5. Avoid the most common mistake: using a random train/test split on time-series data

---

**Week 12 Theme:** Airbnb-style listing price prediction

## The Onion Analogy

> *"A booking on Saturday night in December tells you more than just 'day 6 of the week' and 'month 12'. It tells you: weekend (higher demand), peak holiday season, end-of-year travel surge. DateTime features are like peeling an onion — each layer reveals more signal."*

A raw timestamp like `2024-12-21 22:30:00` looks like a single number to a machine learning model. But it encodes *at least* a dozen meaningful signals:

- **Year:** macro-economic trends, platform growth
- **Month / Quarter:** seasonal demand peaks (summer holidays, Christmas)
- **Week of year:** school holidays, bank holidays
- **Day of week:** weekday vs. weekend demand
- **Hour:** check-in time patterns, surge pricing windows
- **Is month start/end:** salary cycle effects on booking rates

None of this information is accessible to a model that receives the raw integer timestamp. Feature engineering is how we peel the onion.

## Why Does This Matter for ML?

Datetime columns appear in **almost every real-world dataset**:

| Domain | Datetime column | Signal hiding inside |
|--------|----------------|---------------------|
| E-commerce | `order_timestamp` | Peak hours, seasonal demand |
| Finance | `transaction_date` | Month-end effects, weekday patterns |
| IoT / sensors | `reading_time` | Day/night cycles, shift patterns |
| Healthcare | `appointment_date` | Seasonal illness peaks |
| Airbnb | `booking_date` | Weekends, holidays, travel seasons |

A model that receives only `booking_date` as a raw integer has **no chance** of learning these patterns — the integer representation makes December 25th look like it is close to December 24th but very far from December 25th the previous year. Explicit feature extraction makes the model's job dramatically easier.

**Bottom line:** datetime feature engineering is often the highest-ROI activity for any time-aware dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)                          # reproducibility
sns.set_theme(style='whitegrid')            # consistent plot styling

# ── Generate synthetic Airbnb bookings dataset ──────────────────────────────
N = 2000                                    # total bookings
N_LISTINGS = 50                             # number of unique listings

# 2 years of daily dates starting 2022-01-01
date_range = pd.date_range(start='2022-01-01', periods=730, freq='D')
booking_dates = np.random.choice(date_range, size=N, replace=True)

listing_ids = np.random.randint(1, N_LISTINGS + 1, size=N)  # 50 listings

# Price has a seasonal component (higher in summer/winter) + weekend bump
day_of_week   = pd.to_datetime(booking_dates).dayofweek
month_vals    = pd.to_datetime(booking_dates).month
seasonal      = 20 * np.sin(2 * np.pi * month_vals / 12)   # sinusoidal seasonal trend
weekend_bump  = 15 * (day_of_week >= 5).astype(int)         # weekend premium
listing_base  = listing_ids * 1.5                            # each listing has own base price
noise         = np.random.normal(0, 20, size=N)

price = 80 + listing_base + seasonal + weekend_bump + noise
price = np.clip(price, 40, 400)            # realistic price range

num_guests = np.random.randint(1, 7, size=N)

df = pd.DataFrame({
    'booking_date': pd.to_datetime(booking_dates),
    'listing_id':   listing_ids,
    'num_guests':   num_guests,
    'price':        price.round(2),
})

# Sort by listing then date — important for lag/rolling features later
df = df.sort_values(['listing_id', 'booking_date']).reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df.booking_date.min().date()} to {df.booking_date.max().date()}')
df.head(10)

## 1. Basic Datetime Extraction

Pandas `.dt` accessor gives us instant access to every component of a datetime. The table below shows the most useful attributes:

| Attribute | Returns | Example |
|-----------|---------|--------|
| `.dt.year` | Integer year | 2023 |
| `.dt.month` | 1–12 | 6 |
| `.dt.day` | 1–31 | 15 |
| `.dt.dayofweek` | 0 (Mon) – 6 (Sun) | 4 |
| `.dt.quarter` | 1–4 | 2 |
| `.dt.isocalendar().week` | ISO week number 1–53 | 24 |
| `.dt.is_month_start` | Boolean | False |
| `.dt.is_month_end` | Boolean | False |

We wrap this in a reusable function so we can call it on any DataFrame.

In [ ]:
np.random.seed(42)

def extract_datetime_features(df, date_col):
    """Extract a rich set of features from a datetime column.
    
    Parameters
    ----------
    df       : pd.DataFrame
    date_col : str  — name of the datetime column
    
    Returns
    -------
    pd.DataFrame of extracted features (same row order as input)
    """
    dt = df[date_col]                            # shorthand alias
    features = pd.DataFrame({
        'year':           dt.dt.year,
        'month':          dt.dt.month,           # 1 = January … 12 = December
        'day':            dt.dt.day,             # day of month
        'dayofweek':      dt.dt.dayofweek,       # 0 = Monday … 6 = Sunday
        'quarter':        dt.dt.quarter,         # 1–4
        'weekofyear':     dt.dt.isocalendar().week.astype(int),  # ISO week
        'is_weekend':     (dt.dt.dayofweek >= 5).astype(int),    # Saturday/Sunday → 1
        'is_month_start': dt.dt.is_month_start.astype(int),      # 1st of month → 1
        'is_month_end':   dt.dt.is_month_end.astype(int),        # last day → 1
    })
    return features

# Apply the function to our dataset
dt_features = extract_datetime_features(df, 'booking_date')

print('Extracted feature matrix shape:', dt_features.shape)
print('\nSample (first 5 rows):')
dt_features.head()

## 2. The Circularity Problem

When we use raw integer encoding for day-of-week (0 = Monday, 6 = Sunday), a linear model sees:

- Monday = 0
- Sunday = 6
- Distance between them = **6 units**

But from a *business* perspective, Sunday night and Monday morning are **adjacent** — they are both part of the weekend/weekday transition. The raw encoding places them at opposite ends of the scale.

This is the **circularity problem**: any periodic variable (hours 0–23, months 1–12, days 0–6) wraps around, but integer encoding does not.

The plot below makes the problem visible: the price signal is continuous around the week cycle, but the raw encoding creates an artificial discontinuity between 6 (Sunday) and 0 (Monday).

In [ ]:
np.random.seed(42)

# Compute mean price per day-of-week (0=Mon … 6=Sun)
df['dayofweek'] = df['booking_date'].dt.dayofweek
mean_by_dow = df.groupby('dayofweek')['price'].mean()

day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: raw integer encoding — the line drops sharply from Sun(6) back to Mon(0)
axes[0].plot(mean_by_dow.index, mean_by_dow.values, marker='o', color='steelblue', linewidth=2)
axes[0].set_xticks(range(7))
axes[0].set_xticklabels(day_labels)
axes[0].set_title('Raw Integer Encoding\n(discontinuity: Sun→Mon gap is 6 units)', fontsize=11)
axes[0].set_xlabel('Day of Week (integer)')
axes[0].set_ylabel('Mean Price ($)')
axes[0].annotate('Artificial\ndrop here', xy=(6, mean_by_dow[6]),
                 xytext=(4.5, mean_by_dow[6] - 8),
                 arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=9)

# Right: circular (wrapped) view — Sun and Mon should sit next to each other
# We duplicate Mon at position 7 to show the continuous cycle
extended_index  = list(mean_by_dow.index) + [7]
extended_values = list(mean_by_dow.values) + [mean_by_dow[0]]  # wrap Mon back
extended_labels = day_labels + ['Mon (wrap)']

axes[1].plot(extended_index, extended_values, marker='o', color='darkorange', linewidth=2)
axes[1].set_xticks(range(8))
axes[1].set_xticklabels(extended_labels, rotation=15)
axes[1].set_title('True Circular Pattern\n(Sun and Mon are adjacent!)', fontsize=11)
axes[1].set_xlabel('Day of Week (wrapped)')
axes[1].set_ylabel('Mean Price ($)')

plt.suptitle('The Circularity Problem in Day-of-Week Encoding', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('The raw encoding makes Sunday (6) look FAR from Monday (0).')
print('Cyclical encoding will fix this by mapping both to adjacent points on a circle.')

## 3. Cyclical Encoding with Sin / Cos

**The idea:** map each value in a periodic variable onto a point on the unit circle using:

$$\text{sin\_enc} = \sin\!\left(\frac{2\pi \cdot d}{\text{period}}\right), \quad \text{cos\_enc} = \cos\!\left(\frac{2\pi \cdot d}{\text{period}}\right)$$

For day-of-week (period = 7):
- Monday (0)  → (sin=0, cos=1)
- Sunday (6)  → (sin≈−0.78, cos≈0.62)
- Monday (7)  → (sin=0, cos=1)  ← same as Monday 0 ✓

Sunday and Monday now sit **next to each other** on the circle — the model can discover this adjacency using any distance-based computation.

**Why two columns (sin + cos)?** A single sin column is ambiguous: sin(30°) = sin(150°). You need both to uniquely identify every position on the circle.

In [ ]:
np.random.seed(42)

def cyclical_encode(series, period):
    """Encode a periodic integer series using sin and cos projections.
    
    Parameters
    ----------
    series : array-like of integers (e.g., 0–6 for day-of-week)
    period : int — full cycle length (7 for days, 12 for months, 24 for hours)
    
    Returns
    -------
    (sin_values, cos_values)  — two arrays, same length as series
    """
    angle = 2 * np.pi * np.array(series) / period  # angle in radians
    return np.sin(angle), np.cos(angle)

# ── Apply cyclical encoding to key periodic columns ─────────────────────────
df['month'] = df['booking_date'].dt.month          # 1–12
df['hour']  = 12                                   # synthetic: assume midday bookings

df['dow_sin'], df['dow_cos']     = cyclical_encode(df['dayofweek'], period=7)
df['month_sin'], df['month_cos'] = cyclical_encode(df['month'],     period=12)
df['hour_sin'], df['hour_cos']   = cyclical_encode(df['hour'],      period=24)

# ── Visualisation: unit circles for dayofweek and month ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

plot_specs = [
    ('Day of Week',  'dow_sin',   'dow_cos',   7,  ['M','T','W','Th','F','Sa','Su']),
    ('Month',        'month_sin', 'month_cos', 12, ['J','F','M','A','M','J','J','A','S','O','N','D']),
    ('Hour (0-23)',  'hour_sin',  'hour_cos',  24, None),
]

for ax, (title, scol, ccol, period, labels) in zip(axes, plot_specs):
    # Draw unit circle guide
    theta = np.linspace(0, 2*np.pi, 300)
    ax.plot(np.cos(theta), np.sin(theta), color='lightgray', linewidth=1, zorder=0)

    # Compute unique positions for this period
    vals   = np.arange(period)
    s, c   = cyclical_encode(vals, period)
    ax.scatter(c, s, s=80, zorder=3, color='steelblue')

    if labels:
        for i, lbl in enumerate(labels):
            ax.annotate(lbl, (c[i]+0.05, s[i]+0.05), fontsize=8)

    # Draw lines from centre to show adjacency
    for i in range(period):
        ax.plot([c[i], c[(i+1) % period]], [s[i], s[(i+1) % period]],
                color='darkorange', alpha=0.4, linewidth=1)

    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_aspect('equal')
    ax.set_title(f'Cyclical Encoding: {title}', fontsize=11)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_xlabel('cos component')
    ax.set_ylabel('sin component')

plt.suptitle('Consecutive values are ADJACENT on the circle — no discontinuity', fontsize=12)
plt.tight_layout()
plt.show()

# Show sample encoded values
print('\nSample cyclical encoding for day-of-week:')
sample = pd.DataFrame({'dayofweek': range(7),
                        'day_name':  ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']})
sample['dow_sin'], sample['dow_cos'] = cyclical_encode(sample['dayofweek'], 7)
print(sample.round(3).to_string(index=False))

In [ ]:
np.random.seed(42)

# ── Empirical comparison: raw vs one-hot vs cyclical encoding ────────────────
# Build three feature matrices and compare cross-validated RMSE on price prediction

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

y = df['price'].values

# (a) Raw integer encoding: dayofweek as single integer 0-6
X_raw = df[['dayofweek', 'month', 'is_weekend']].values

# (b) One-hot encoding of dayofweek (7 columns) + month (12 columns)
ohe = OneHotEncoder(sparse_output=False)
X_ohe_dow   = ohe.fit_transform(df[['dayofweek']])  # 7 columns
X_ohe_month = ohe.fit_transform(df[['month']])      # 12 columns
X_ohe = np.hstack([X_ohe_dow, X_ohe_month])

# (c) Cyclical sin+cos encoding: 2 columns for dow + 2 for month = 4 columns
X_cyc = df[['dow_sin', 'dow_cos', 'month_sin', 'month_cos']].values

def rmse_cv(X, y, cv=5):
    """Return mean RMSE from cross-validation with LinearRegression."""
    lr  = LinearRegression()
    # Negative MSE scores → take sqrt → RMSE
    mse_scores = -cross_val_score(lr, X, y, cv=cv, scoring='neg_mean_squared_error')
    return np.sqrt(mse_scores).mean()

results = {
    'Raw integer (3 features)':    rmse_cv(X_raw, y),
    'One-hot (19 features)':        rmse_cv(X_ohe, y),
    'Cyclical sin/cos (4 features)': rmse_cv(X_cyc, y),
}

print('Cross-validated RMSE comparison (lower is better):\n')
for name, rmse in results.items():
    bar = '█' * int(rmse / 2)
    print(f'  {name:<35} RMSE = {rmse:.2f}  {bar}')

print('\n→ Cyclical encoding matches or beats one-hot with far fewer features.')
print('  Raw integer encoding is worst because linear models cannot handle circularity.')

## 4. Lag Features

**Lag features** answer the question: *"What was the price N days ago?"*

They are powerful because prices exhibit **temporal autocorrelation** — yesterday's price is a strong predictor of today's. This captures:

- **Short-term momentum:** if a listing raised prices last week, it might be raising them this week too
- **Weekly seasonality:** `price_lag_7d` encodes the price on the same day last week — directly comparable
- **Monthly patterns:** `price_lag_30d` captures the "same time last month" effect

**Critical step:** lags must be computed *within each listing group* — the price 7 days ago for listing 3 should not be confused with the price 7 days ago for listing 17.

In [ ]:
np.random.seed(42)

# ── Create lag features per listing ─────────────────────────────────────────
# Already sorted by ['listing_id', 'booking_date'] above
df_sorted = df.copy()

for lag in [1, 7, 14, 30]:                     # 1-day, weekly, 2-week, monthly lags
    df_sorted[f'price_lag_{lag}d'] = (
        df_sorted
        .groupby('listing_id')['price']         # separate lag per listing
        .shift(lag)                              # shift by `lag` rows within the group
    )

# ── Check correlation of each lag with current price ────────────────────────
lag_cols = [f'price_lag_{l}d' for l in [1, 7, 14, 30]]
correlations = df_sorted[lag_cols + ['price']].corr()['price'].drop('price')

print('Correlation of lag features with current price:')
for col, corr in correlations.items():
    bar = '█' * int(abs(corr) * 30)
    print(f'  {col:<20} r = {corr:.3f}  {bar}')

# ── Plot autocorrelation for one listing ────────────────────────────────────
from pandas.plotting import autocorrelation_plot

listing_prices = (df_sorted[df_sorted['listing_id'] == 1]
                  .sort_values('booking_date')['price'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: raw price time series for one listing
axes[0].plot(listing_prices.values[:100], color='steelblue', linewidth=1.2)
axes[0].set_title('Price Time Series — Listing 1 (first 100 bookings)', fontsize=11)
axes[0].set_xlabel('Booking index')
axes[0].set_ylabel('Price ($)')

# Right: lag scatter — price_lag_7d vs price (should cluster along diagonal)
subset = df_sorted[df_sorted['listing_id'] == 1].dropna(subset=['price_lag_7d'])
axes[1].scatter(subset['price_lag_7d'], subset['price'],
                alpha=0.4, s=20, color='darkorange')
# Perfect-prediction diagonal
lims = [subset['price_lag_7d'].min(), subset['price_lag_7d'].max()]
axes[1].plot(lims, lims, 'k--', linewidth=1, label='y = x (perfect lag prediction)')
axes[1].set_title('7-Day Lag vs Current Price (Listing 1)', fontsize=11)
axes[1].set_xlabel('price_lag_7d')
axes[1].set_ylabel('Current price')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nRows with NaN lags (first {lag} bookings per listing have no history):",
      df_sorted['price_lag_7d'].isna().sum())

## 5. Rolling Window Features

Lag features give you a *single* historical snapshot. **Rolling window features** give you a *summary* over a recent window:

- **Rolling mean:** smooth price trend over the last N days
- **Rolling std:** price volatility — high std = inconsistent pricing strategy
- **Rolling max:** captures recent price ceiling

A 7-day rolling mean is particularly powerful: it encodes "what the price normally is at this time of week" which is exactly the seasonality pattern we care about.

> **min_periods=1** means we compute the mean even with fewer than `window` observations — useful at the start of each listing's history.

In [ ]:
np.random.seed(42)

# ── Rolling window features per listing ─────────────────────────────────────
for window in [7, 14, 30]:                          # weekly, biweekly, monthly windows
    rolled = (
        df_sorted
        .groupby('listing_id')['price']
        .rolling(window, min_periods=1)              # allow partial windows at start
    )
    df_sorted[f'price_roll_mean_{window}d'] = rolled.mean().values
    df_sorted[f'price_roll_std_{window}d']  = rolled.std().values

# ── Correlation of rolling features with current price ──────────────────────
roll_cols = [f'price_roll_mean_{w}d' for w in [7, 14, 30]] + \
            [f'price_roll_std_{w}d'  for w in [7, 14, 30]]

roll_corr = df_sorted[roll_cols + ['price']].corr()['price'].drop('price').sort_values(ascending=False)

print('Correlation of rolling features with current price:')
for col, corr in roll_corr.items():
    bar = '█' * int(abs(corr) * 40)
    print(f'  {col:<30} r = {corr:.3f}  {bar}')

# ── Visualise rolling mean vs actual for one listing ────────────────────────
listing1 = df_sorted[df_sorted['listing_id'] == 1].sort_values('booking_date').head(80)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(len(listing1)), listing1['price'].values,
        alpha=0.5, label='Actual price', color='steelblue')
ax.plot(range(len(listing1)), listing1['price_roll_mean_7d'].values,
        linewidth=2.5, label='7-day rolling mean', color='darkorange')
ax.plot(range(len(listing1)), listing1['price_roll_mean_30d'].values,
        linewidth=2, label='30-day rolling mean', color='forestgreen', linestyle='--')
ax.set_title('Rolling Mean Smooths Noise and Reveals Trend — Listing 1', fontsize=11)
ax.set_xlabel('Booking index')
ax.set_ylabel('Price ($)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nThe 7-day rolling mean has correlation {roll_corr['price_roll_mean_7d']:.3f} with price.")
print('This is typically the single most predictive datetime-derived feature.')

## 6. Time-Since Features

**Time-since features** encode *recency*:

- `days_since_host_joined` → host experience (longer = more reliable, better pricing)
- `days_since_last_review` → listing activity (recent reviews = active listing)
- `days_since_last_price_change` → pricing dynamism signal

These work well because many real-world effects are **logarithmic in time** — the difference between 1 day and 10 days is huge; between 1000 and 1010 days is negligible. We often apply `np.log1p()` to these features.

In [ ]:
np.random.seed(42)

# ── Simulate host_joined_date and last_review_date ───────────────────────────
# Each listing has a host who joined 1-2000 days before 2022-01-01
host_join_offsets  = np.random.randint(30, 2000, size=N)   # days before dataset start
review_age_offsets = np.random.randint(0, 180, size=N)     # last review 0-180 days ago

ref_date = pd.Timestamp('2022-01-01')
df_sorted['host_joined_date'] = ref_date - pd.to_timedelta(host_join_offsets, unit='D')
df_sorted['last_review_date'] = df_sorted['booking_date'] - pd.to_timedelta(review_age_offsets, unit='D')

# ── Compute time-since features ──────────────────────────────────────────────
df_sorted['days_since_host_joined'] = (
    df_sorted['booking_date'] - df_sorted['host_joined_date']
).dt.days

df_sorted['days_since_last_review'] = (
    df_sorted['booking_date'] - df_sorted['last_review_date']
).dt.days

# Log-transform: diminishing returns as time increases
df_sorted['log_days_host']   = np.log1p(df_sorted['days_since_host_joined'])
df_sorted['log_days_review'] = np.log1p(df_sorted['days_since_last_review'])

print('Correlation with price:')
for col in ['days_since_host_joined', 'log_days_host',
            'days_since_last_review', 'log_days_review']:
    corr = df_sorted['price'].corr(df_sorted[col])
    print(f'  {col:<35} r = {corr:.3f}')

# ── Box plots of price by days-since-last-review bins ───────────────────────
df_sorted['review_recency_bin'] = pd.cut(
    df_sorted['days_since_last_review'],
    bins=[0, 7, 30, 90, 180, 999],
    labels=['0-7d', '8-30d', '31-90d', '91-180d', '>180d']
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df_sorted, x='review_recency_bin', y='price',
            palette='Blues_d', ax=ax, order=['0-7d', '8-30d', '31-90d', '91-180d', '>180d'])
ax.set_title('Price Distribution by Days Since Last Review\n'
             '(Recent reviews → higher listing quality → premium price)', fontsize=11)
ax.set_xlabel('Days since last review')
ax.set_ylabel('Booking price ($)')
plt.tight_layout()
plt.show()

## 7. Holiday and Domain-Specific Event Features

Generic calendar features (month, dayofweek) miss specific high-impact events:

- **Public holidays:** Christmas, New Year, Easter → massive demand spikes
- **Local events:** concerts, sporting events, festivals → hyper-local surges
- **School holidays:** family travel peaks

For Airbnb, proximity to major holidays is one of the strongest predictors of price premiums — often adding 30–50% to base price.

We encode two features:
1. `is_holiday` — binary flag for the holiday itself
2. `days_to_nearest_holiday` — continuous distance; captures the *lead-up* effect

In [ ]:
np.random.seed(42)

# ── Define 12 major UK/US holidays (two years) ──────────────────────────────
holidays = pd.to_datetime([
    '2022-01-01', '2022-04-15', '2022-05-30', '2022-07-04',
    '2022-09-05', '2022-11-24', '2022-12-25', '2022-12-31',
    '2023-01-01', '2023-04-07', '2023-07-04', '2023-12-25',
])

holiday_set = set(holidays.date)  # fast O(1) lookup

# Binary flag: 1 if the booking date is itself a holiday
df_sorted['is_holiday'] = df_sorted['booking_date'].dt.date.isin(holiday_set).astype(int)

def days_to_nearest_holiday(dates, holidays):
    """Return the number of days to the nearest holiday for each date."""
    holidays_arr = np.array(holidays, dtype='datetime64[D]')  # convert once
    result = []
    for d in dates:
        diffs = np.abs((holidays_arr - np.datetime64(d, 'D')).astype(int))
        result.append(diffs.min())                               # nearest holiday distance
    return np.array(result)

df_sorted['days_to_holiday'] = days_to_nearest_holiday(
    df_sorted['booking_date'].values, holidays
)

print('Holiday feature stats:')
print(f'  Holiday bookings: {df_sorted["is_holiday"].sum()} / {len(df_sorted)}')
print(f'  Mean days to nearest holiday: {df_sorted["days_to_holiday"].mean():.1f}')

# ── Violin plot: price around holidays vs regular dates ──────────────────────
df_sorted['proximity_bin'] = pd.cut(
    df_sorted['days_to_holiday'],
    bins=[-1, 0, 3, 7, 14, 999],
    labels=['Holiday itself', '1-3 days away', '4-7 days away',
            '8-14 days away', 'Regular date']
)

fig, ax = plt.subplots(figsize=(11, 5))
order = ['Holiday itself', '1-3 days away', '4-7 days away', '8-14 days away', 'Regular date']
sns.violinplot(data=df_sorted, x='proximity_bin', y='price',
               palette='OrRd_r', order=order, ax=ax, inner='box')
ax.set_title('Price Distribution by Proximity to Nearest Holiday\n'
             '(Holiday periods command premium pricing)', fontsize=11)
ax.set_xlabel('Proximity to nearest holiday')
ax.set_ylabel('Booking price ($)')
ax.tick_params(axis='x', rotation=10)
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# ── Final comparison: baseline vs full datetime feature set ──────────────────
# Baseline: just listing_id and num_guests
# Enhanced: + all datetime features we created

from sklearn.ensemble import RandomForestRegressor

feature_cols = [
    'listing_id', 'num_guests',
    'year', 'month', 'day', 'dayofweek', 'quarter', 'weekofyear',
    'is_weekend', 'is_month_start', 'is_month_end',
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'price_lag_7d', 'price_lag_14d', 'price_lag_30d',
    'price_roll_mean_7d', 'price_roll_mean_14d', 'price_roll_std_7d',
    'log_days_host', 'log_days_review',
    'is_holiday', 'days_to_holiday',
]

# Drop rows with any NaN (lag features are NaN for early rows)
df_model = df_sorted[feature_cols + ['price']].dropna()

X_baseline  = df_model[['listing_id', 'num_guests']].values
X_full      = df_model[feature_cols].values
y_model     = df_model['price'].values

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

def r2_cv(X, y, cv=5):
    scores = cross_val_score(rf, X, y, cv=cv, scoring='r2')
    return scores.mean(), scores.std()

r2_base, std_base   = r2_cv(X_baseline, y_model)
r2_full, std_full   = r2_cv(X_full,     y_model)

print(f'Baseline  R²: {r2_base:.3f} ± {std_base:.3f}')
print(f'Full DT   R²: {r2_full:.3f} ± {std_full:.3f}')
print(f'Improvement:  +{r2_full - r2_base:.3f}')

# ── Feature importances from the full model ──────────────────────────────────
rf.fit(X_full, y_model)                     # fit on all data for importance plot
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
top15 = importances.tail(15)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['darkorange' if 'roll' in c or 'lag' in c or 'holiday' in c
          else 'steelblue' for c in top15.index]
top15.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Top 15 Feature Importances (RandomForest)\n'
             'Orange = datetime-derived features', fontsize=11)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print('\nTop 5 most important features:')
for feat, imp in importances.tail(5).iloc[::-1].items():
    print(f'  {feat:<30} {imp:.4f}')

## 8. The Temporal Split Warning

For time-series data, a **random train/test split is almost always wrong**.

Here is why: if we randomly pick 20% of rows as test data, those rows are *interspersed* with training rows. Our lag and rolling features are computed using data from *both before and after* each test point — meaning the model has seen the future during training. This is **data leakage**.

**Always use a temporal split** for time-series data:
- Train on data *before* a cutoff date
- Test on data *after* the cutoff date

For cross-validation, use `TimeSeriesSplit` from sklearn — it ensures each fold's test data is always in the future relative to its training data.

## Self-Check Questions

Test your understanding before moving on.

---

**Q1.** Why does cyclical encoding use *both* sin and cos rather than just sin?

<details><summary>Show answer</summary>

A single sin value is ambiguous: `sin(30°) = sin(150°)`, so two different points on the cycle map to the same value. By using both sin and cos together, every position on the cycle maps to a **unique (sin, cos) pair** — just like every point on the unit circle is uniquely defined by its (x, y) coordinates.

</details>

---

**Q2.** You are computing a 7-day rolling mean for lag features. Your dataset has 3 bookings per listing on day 1. What does `min_periods=1` do, and is it the right choice?

<details><summary>Show answer</summary>

`min_periods=1` means: if the window contains at least 1 observation, compute the mean (rather than returning NaN). This is usually the right choice for the early part of a time series so you do not discard the first N-1 rows. However, you should be aware that the rolling mean over 1 observation is just that observation itself — not a very stable estimate. For inference, document this behaviour.

</details>

---

**Q3.** A colleague argues: "I used `cross_val_score` with `cv=5` on my booking price model and got R² = 0.92. This is amazing!" What concern should you raise?

<details><summary>Show answer</summary>

The default `KFold` cross-validation in `cross_val_score` **shuffles data randomly** before splitting. For a time-series dataset with lag and rolling features, this creates **data leakage**: test fold rows have future data in their training fold. The R² = 0.92 is likely inflated. They should use `TimeSeriesSplit` and recompute lag/rolling features within each fold to get a realistic performance estimate.

</details>

## Key Takeaways

1. **Always use cyclical encoding for periodic features** (hour, dayofweek, month). Raw integers create an artificial discontinuity at the wrap-around point that harms linear models.

2. **Rolling and lag features are the most powerful datetime signals.** The 7-day rolling mean typically achieves correlation > 0.9 with the current price, outperforming all pure calendar features.

3. **Use a temporal split** — never random — for time-series data. Use `TimeSeriesSplit` for cross-validation. Computing lag/rolling features on the full dataset before splitting leaks future information.

4. **Holiday proximity features** can add 5–15% predictive power for booking/retail/transport datasets. Always check the domain calendar.

5. **Log-transform time-since features** (`np.log1p`). The effect of experience is diminishing — the difference between 1 and 10 days matters far more than the difference between 1000 and 1010 days.

---

*Next up: **05 — Text Feature Engineering** — extracting signals from Airbnb listing descriptions.*